In [1]:
from agentic.infrastructure.app_configuration.model.configuration import AppConfiguration
from agentic.bootstrap.container import Container
from agentic.infrastructure.app_configuration.model.connector import DatabaseConnector

container = Container()
is_on, exception = await container.boot
configuration: AppConfiguration = container.application_configuration

In [2]:
from agentic.infrastructure.repository.repository import RepositoryFactory, AsyncSQLRepository
from agentic.infrastructure.repository.sqlite.factory import SQLiteRepositoryFactory
from agentic.infrastructure.repository.sqlite.mapper import SettingsMapper
from typing import cast
from agentic.infrastructure.repository.sqlite.settings import SqliteSettings
from agentic.infrastructure.app_configuration.enum.connector_type import ConnectorType

sqlite_connector: DatabaseConnector = cast(
    DatabaseConnector, configuration.connector.get(ConnectorType.database).get("sqlite")
)
sqlite_settings = SqliteSettings(SettingsMapper(sqlite_connector)())
sqlite_factory: RepositoryFactory = SQLiteRepositoryFactory(sqlite_settings)
sqlite_repository: AsyncSQLRepository = await sqlite_factory.create_repository()

In [ ]:
from random import choice, randint, uniform
from faker import Faker

fake = Faker()
# =============================================================================
# USER SERVICE DASHBOARD
# =============================================================================

await sqlite_repository.execute("""
CREATE TABLE IF NOT EXISTS user_service_dashboard (
    user_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    account_status TEXT NOT NULL,
    subscription_plan TEXT NOT NULL,
    country TEXT NOT NULL,
    created_at TEXT NOT NULL,
    last_login TEXT,
    failed_login_attempts INTEGER NOT NULL
)
""")

for i in range(100):
    await sqlite_repository.execute(
        """
        INSERT INTO user_service_dashboard
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            i + 1,
            fake.name(),
            fake.unique.email(),
            choice(["ACTIVE", "LOCKED", "SUSPENDED"]),
            choice(["FREE", "PRO", "ENTERPRISE"]),
            fake.country(),
            fake.iso8601(),
            fake.iso8601(),
            randint(0, 5),
        ),
    )


# =============================================================================
# PAYMENT SERVICE DASHBOARD
# =============================================================================

await sqlite_repository.execute("""
CREATE TABLE IF NOT EXISTS payment_service_dashboard (
    payment_id INTEGER PRIMARY KEY,
    customer_email TEXT NOT NULL,
    amount REAL NOT NULL,
    currency TEXT NOT NULL,
    payment_method TEXT NOT NULL,
    payment_status TEXT NOT NULL,
    provider TEXT NOT NULL,
    processed_at TEXT NOT NULL
)
""")

for i in range(100):
    await sqlite_repository.execute(
        """
        INSERT INTO payment_service_dashboard
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            i + 1,
            fake.email(),
            round(uniform(10, 2500), 2),
            choice(["USD", "EUR", "GBP"]),
            choice(["CARD", "PAYPAL", "APPLE_PAY"]),
            choice(["SUCCESS", "FAILED", "PENDING", "REFUNDED"]),
            choice(["Stripe", "Adyen", "Checkout"]),
            fake.iso8601(),
        ),
    )


# =============================================================================
# INVENTORY SERVICE DASHBOARD
# =============================================================================

await sqlite_repository.execute("""
CREATE TABLE IF NOT EXISTS inventory_service_dashboard (
    product_id INTEGER PRIMARY KEY,
    sku TEXT NOT NULL,
    product_name TEXT NOT NULL,
    warehouse TEXT NOT NULL,
    stock_quantity INTEGER NOT NULL,
    reorder_threshold INTEGER NOT NULL,
    supplier TEXT NOT NULL,
    last_restock TEXT
)
""")

for i in range(100):
    await sqlite_repository.execute(
        """
        INSERT INTO inventory_service_dashboard
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            i + 1,
            fake.unique.bothify("SKU-#####"),
            fake.word().title(),
            choice(["Paris", "Berlin", "New York", "Tokyo"]),
            randint(0, 500),
            randint(10, 100),
            fake.company(),
            fake.iso8601(),
        ),
    )


# =============================================================================
# ORDER SERVICE DASHBOARD
# =============================================================================

await sqlite_repository.execute("""
CREATE TABLE IF NOT EXISTS order_service_dashboard (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    order_status TEXT NOT NULL,
    total_amount REAL NOT NULL,
    shipping_country TEXT NOT NULL,
    items_count INTEGER NOT NULL,
    priority TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")

for i in range(100):
    await sqlite_repository.execute(
        """
        INSERT INTO order_service_dashboard
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            i + 1,
            fake.name(),
            choice(
                [
                    "CREATED",
                    "PAID",
                    "SHIPPED",
                    "DELIVERED",
                    "CANCELLED",
                ]
            ),
            round(uniform(20, 1500), 2),
            fake.country(),
            randint(1, 8),
            choice(["LOW", "NORMAL", "HIGH"]),
            fake.iso8601(),
        ),
    )


# =============================================================================
# INCIDENT DASHBOARD
# =============================================================================

await sqlite_repository.execute("""
CREATE TABLE IF NOT EXISTS incident_dashboard (
    incident_id INTEGER PRIMARY KEY,
    service_name TEXT NOT NULL,
    severity TEXT NOT NULL,
    environment TEXT NOT NULL,
    status TEXT NOT NULL,
    owner_team TEXT NOT NULL,
    opened_at TEXT NOT NULL,
    resolved_at TEXT,
    affected_users INTEGER
)
""")

for i in range(100):
    await sqlite_repository.execute(
        """
        INSERT INTO incident_dashboard
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            i + 1,
            choice(
                [
                    "user-service",
                    "payment-service",
                    "inventory-service",
                    "order-service",
                    "notification-service",
                ]
            ),
            choice(["P1", "P2", "P3", "P4"]),
            choice(["prod", "staging"]),
            choice(["OPEN", "INVESTIGATING", "RESOLVED"]),
            choice(
                [
                    "Platform",
                    "Payments",
                    "Orders",
                    "Identity",
                    "Inventory",
                ]
            ),
            fake.iso8601(),
            fake.iso8601() if randint(0, 1) else None,
            randint(0, 250_000),
        ),
    )

print("✅ Created 5 dashboard tables")
print("✅ Inserted 500 rows of realistic microservice data")

In [4]:
await sqlite_repository.execute("SELECT * FROM user_service_dashboard")

[{'user_id': 1,
  'full_name': 'Daniel Perez',
  'email': 'mmack@example.com',
  'account_status': 'LOCKED',
  'subscription_plan': 'ENTERPRISE',
  'country': 'Togo',
  'created_at': '1972-04-30T09:14:05.121620',
  'last_login': '1986-10-06T18:29:49.677482',
  'failed_login_attempts': 2},
 {'user_id': 2,
  'full_name': 'Elizabeth Andrews',
  'email': 'cobbsavannah@example.com',
  'account_status': 'LOCKED',
  'subscription_plan': 'ENTERPRISE',
  'country': 'Botswana',
  'created_at': '2002-03-01T14:29:51.520455',
  'last_login': '1986-04-10T01:46:18.970576',
  'failed_login_attempts': 2},
 {'user_id': 3,
  'full_name': 'Christine Simpson',
  'email': 'teresarodriguez@example.org',
  'account_status': 'SUSPENDED',
  'subscription_plan': 'ENTERPRISE',
  'country': 'Seychelles',
  'created_at': '2025-10-31T23:11:05.828642',
  'last_login': '2022-10-02T13:54:11.769787',
  'failed_login_attempts': 1},
 {'user_id': 4,
  'full_name': 'Arthur Hughes',
  'email': 'lmontoya@example.net',
  'acco